Send Forecast Email
Reads the latest generated forecast and emails a readable summary to the ERP team for validation.

**Input**: gold/live/forecasts/overall_forecast_latest.json
**Output**: email sent

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service
import json
import datetime

blob_service = get_blob_service(storage_account_name, storage_account_key)

FORECAST_BASE = "live/battery"

Load active overall forecasts

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_monthly = json.loads(stream)

print(active_weekly)
print(active_monthly)

Load active brand forecasts

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/brand_weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_brand_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/brand_monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_brand_monthly = json.loads(stream)

print(active_brand_weekly)
print(active_brand_monthly)

Load active vehicle forecasts

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/vehicle_weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_vehicle_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/vehicle_monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_vehicle_monthly = json.loads(stream)

print(active_vehicle_weekly)
print(active_vehicle_monthly)

Load active location forecasts

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/location_weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_location_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/location_monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_location_monthly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/district_weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_district_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/district_monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_district_monthly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/province_weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_province_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/province_monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_province_monthly = json.loads(stream)


print(active_location_weekly)
print(active_location_monthly)
print(active_district_weekly)
print(active_district_monthly)
print(active_province_monthly)

Build the email body

In [0]:
body = """Hello,

Here is the latest battery sales forecast.

WEEKLY FORECAST
"""
for w in active_weekly:
    body += f"Week starting {w['week_start']}: {w['predicted_units']:,} units (generated {w['generated_date']})\n"

body += "\nMONTHLY FORECAST\n"
for m in active_monthly:
    body += f"{m['month_start']}: {m['predicted_units']:,} units (range: {m['lower_bound']:,} - {m['upper_bound']:,}, generated {m['generated_date']})\n"

body += "\nBRAND BREAKDOWN (Weekly)\n"
for w in active_brand_weekly:
    body += f"{w['brand_code']} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += "\nBRAND BREAKDOWN (Monthly)\n"
for m in active_brand_monthly:
    body += f"{m['brand_code']} — {m['month_start']}: {m['predicted_units']:,} units\n"

body += "\nVEHICLE TYPE BREAKDOWN (Weekly)\n"
for w in active_vehicle_weekly:
    body += f"{w['vehicle_type']} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += "\nVEHICLE TYPE BREAKDOWN (Monthly)\n"
for m in active_vehicle_monthly:
    body += f"{m['vehicle_type']} — {m['month_start']}: {m['predicted_units']:,} units\n"

body += "\nLOCATION BREAKDOWN (Weekly)\n"
for w in active_location_weekly:
    body += f"{w['location_code'], ["location_description"]} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += "\nLOCATION BREAKDOWN (Monthly)\n"
for m in active_location_monthly:
    body += f"{m['location_code'], ["location_description"]} — {m['month_start']}: {m['predicted_units']:,} units\n"

body += "\nLOCATION (DISTRCIT) BREAKDOWN (Weekly)\n"
for w in active_vehicle_weekly:
    body += f"{w['district_name']} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += "\nLOCATION (DISTRCIT) BREAKDOWN (Monthly)\n"
for w in active_vehicle_weekly:
    body += f"{w['district_name']} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += "\nLOCATION (PROVINCE) BREAKDOWN (Weekly)\n"
for m in active_vehicle_monthly:
    body += f"{m['province_name']} — {m['month_start']}: {m['predicted_units']:,} units\n"

body += "\nLOCATION (PROVINCE) BREAKDOWN (Monthly)\n"
for w in active_vehicle_weekly:
    body += f"{w['province_name']} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += """
Please review and let us know if these numbers look reasonable based on your knowledge of current orders/promotions.

This is an automated message from the Exide Sales Forecasting pipeline.
"""

print(body)

Excel attachments

In [0]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
import smtplib

today_str = datetime.date.today().isoformat()

msg = MIMEMultipart()
msg["From"] = smtp_username
msg["To"] = ", ".join(to_emails)
msg["Subject"] = f"Battery Sales Forecast — {today_str}"
msg.attach(MIMEText(body, "plain"))

def attach_excel_from_blob(msg, blob_service, blob_path, filename):
    blob_client = blob_service.get_blob_client(container="gold", blob=blob_path)
    excel_bytes = blob_client.download_blob().readall()
    attachment = MIMEApplication(excel_bytes, _subtype="xlsx")
    attachment.add_header("Content-Disposition", "attachment", filename=filename)
    msg.attach(attachment)

attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/weekly_forecast_history.xlsx", "weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/monthly_forecast_history.xlsx", "monthly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/brand_weekly_forecast_history.xlsx", "brand_weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/brand_monthly_forecast_history.xlsx", "brand_monthly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/vehicle_weekly_forecast_history.xlsx", "vehicle_weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/vehicle_monthly_forecast_history.xlsx", "vehicle_monthly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/location_weekly_forecast_history.xlsx", "location_weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/location_monthly_forecast_history.xlsx", "location_monthly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/district_weekly_forecast_history.xlsx", "district_weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/district_monthly_forecast_history.xlsx", "district_monthly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/province_weekly_forecast_history.xlsx", "province_weekly_forecast_history.xlsx")
attach_excel_from_blob(msg, blob_service, f"{FORECAST_BASE}/forecasts/history/province_monthly_forecast_history.xlsx", "province_monthly_forecast_history.xlsx")


try:
    server = smtplib.SMTP(smtp_server, smtp_port)
    server.starttls()
    server.login(smtp_username, smtp_password)
    server.sendmail(smtp_username, to_emails, msg.as_string())
    print(f"Email sent to {', '.join(to_emails)}")
except Exception as e:
    print(f"Failed to send email: {e}")
finally:
    server.quit()